# Exercise on Joins and anti-joins: add information from other tables

In [1]:
# import libraries - solution
import pandas as pd
import numpy as np

# Set some Pandas options: maximum number of rows/columns it's going to display
#pd.set_option('display.max_rows', 1000)
#pd.set_option('display.max_columns', 100)

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


## Load data from clinical trial

Data comes in two different files. The file `predimed_records.csv` file contains the clinical data for each patient. The file 'predimed_location.csv' contain the information about the meaning of the location codes. Load the two dataframes, inspect them and complete the exercise below.

In [ ]:
# load the dataframes
# solution
df_patients = pd.read_csv('../../data/predimed_records.csv')
df_patients

In [2]:
# solution
df_locations = pd.read_csv('../../data/predimed_location.csv')

df_locations

,location-id,city
0,4,Madrid
1,1,Valencia
2,2,Barcelona
3,5,Bilbao
4,3,Malaga


There were 5 different locations where the study was conducted, each one gave an identification number `patient-id` to each participant.

In [ ]:
# solution
df_locations['location-id'].unique()

## Exercise 1: Add location information (the city) to the patients' records

* For how many patients do we have clinical information? (i.e., rows in `df_patients`)
* Do all patients have an associated location information?

In [ ]:
# solution
len(df_patients)

In [ ]:
# solution
len(df_locations)

In [ ]:
# solution

# Explore the date
len(df_patients['patient-id'].unique())

In [ ]:
# solution

len(df_patients['location-id'].unique())

In [ ]:
# solution

1324*5

Looks like not all patients have been tested at all locations.

#### Solution 1 - O(N*M)

In [ ]:
# %%timeit
# solution

patients_with_city = df_patients.copy()
patients_with_city['city'] = 'n/a'

for idx, row in patients_with_city.iterrows():  # O(N)
    location = row['location-id']
    matching_city = (df_locations['location-id'] == location)   # O(M)
    city = df_locations.loc[matching_city, 'city']
    if len(city) > 0:
        patients_with_city.loc[idx, 'city'] = city.iloc[0]

#### Solution 2 - O(N log N + M log M)
- with sorting

In [ ]:
patients_with_city['city_2'] = 'n/a'

sorted_patients = patients_with_city.sort_values(['location-id'])   # O(N log N)
sorted_locations = df_locations.sort_values(['location-id'])           # O(M log M)

city_2_col = sorted_patients.columns.get_loc('city_2') # get the column location (number)
locations_idx = 0
patients_idx = 0

while True:    # O(N + M)
    row_locations = sorted_locations.iloc[locations_idx]
    key_locations = row_locations['location-id']
    
    row_patients = sorted_patients.iloc[patients_idx]
    key_patients = row_patients['location-id']

    if key_patients == key_locations:
        # print('i am here')
        original_idx = sorted_patients.index[patients_idx]
        patients_with_city.iloc[original_idx, city_2_col] = row_locations['city']
        patients_idx += 1
        # print(patients_idx)
    # elif key_patients < key_locations: # not possible because we've checked that both have 4 unique keys 
    #     # missing data
    #     patients_idx += 1
    else:
        locations_idx += 1
        if locations_idx >= len(sorted_locations):
            break
    if patients_idx >= len(sorted_patients):
        break
    # if patients_idx > 10:
    #     break

In [ ]:
# solution
patients_with_city['city'].equals(patients_with_city['city_2'])

#### Solution - O(n+m)

In [ ]:
# %%timeit
# solution - optimal
df_with_city = patients_with_city.merge(df_locations, on = ['location-id'], how = 'left', suffixes=("", "_3"))

In [ ]:
patients_with_city['city_2'].equals(patients_with_city['city_3'])

In [ ]:
# solution
# same as above but 'by hand'

patients_with_city['city_4'] = 'n/a'
# build hash table: O(M)
city_lookup = {
  (row['location-id']): row['city']
  for _, row in df_locations.iterrows()
}

# probe hash table once per row: O(N)
city_col = [
  city_lookup.get((location), np.nan)
  for location in df_patients['location-id']
]

patients_with_city['city_4'] = city_col

# 4. Save final result in `processed_data_predimed.csv`

1. Using the `.to_csv` method of Pandas DataFrames

In [ ]:
df_without_dropped.to_csv('processed_data_predimed.csv', index=None)